In [3]:
!pip install fastai opencv-python tqdm imutils -q
!pip install wandb

In [6]:
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import cv2
import fastai
from fastai.vision.all import *
from fastai.basics import *
from fastai.callback.all import *
import imutils
import torch
import ast
from pathlib import Path
from google.colab import drive
# from sklearn.model_selection import train_test_split
from fastai.callback.wandb import *

In [ ]:
drive.mount('/content/drive')

### Connect notebook to WanDB

In [ ]:
import wandb
wandb.login(key="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")
wandb.init()

### Load Dataset

In [ ]:
!mkdir data
!cp -v "drive/My Drive/Dartsify/dataset.zip" "data"
!unzip -q -d data data/dataset.zip

dataset_path = Path('data/dataset/')

!cp -v "drive/My Drive/Dartsify/label.csv" "data"

In [ ]:
# Lire les annotations à partir du fichier csv
# ID : nom unique de l'image
# Label : format d'un dictionnaire avec les coordonnées des points de la cible et des fléchettes
# Exemple de Label : "{'objects': {'point': {'x': x1, 'y': y1}}}"
df = pd.read_csv('data/label.csv')[['ID', 'Label']]
df.head(2)

In [ ]:
%%time
# Extraire les coordonnées des pointes des fléchettes du fichier des labels
points = []
for r in df.iterrows():
  d = ast.literal_eval(r[1]['Label'])
  try: x, y = d['objects']['point'].values()
  except: print('Aucun label détecté', r[0])
  points.append((x,y))

df['center_point'] = pd.Series(points)
display(df[['ID', 'center_point']].sample(3))

### Create Masks

In [ ]:
%%time
!mkdir data/masks
mask_path = Path('data/dataset_masks/')

radius = 2
thickness = -1
color = (1, 1, 1)
h, w = 720, 1280

for row in df.iterrows():
  img = np.zeros((h, w, 3), np.uint8)
  x, y = row[1]['center_point']
  center_coordinates = (x, y)
  img2 = cv2.circle(img, center_coordinates, radius, color, thickness)
  mask_filename = row[1]['ID']
  cv2.imwrite(f"{mask_path}/{mask_filename}", cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY))

### Dataloader `get` Functions

In [ ]:
def get_x(dataframe_row):
    return dataset_path/f"{dataframe_row['ID']}"

def get_mask(dataframe_row):
    return mask_path/f"{dataframe_row['ID']}"

### DataBlock



In [ ]:
# Transforms

tfms = [Rotate(), Zoom(), Warp(), Brightness(), Flip(), Contrast(), Normalize.from_stats(*imagenet_stats)]

d_unet = DataBlock(blocks=(ImageBlock, MaskBlock),
                 get_x=get_x,
                 get_y=get_mask,
                 splitter=RandomSplitter(0.1, seed=10),
                 batch_tfms=tfms)

# Batch size est passé dans le dataloader
# Attention à l'overfitting et à l'utilisation excessive de la mémoire GPU 
BATCH_SIZE = 10
# dls pour DataLoaders
dls = d_unet.dataloaders(df, bs=BATCH_SIZE)
xb, yb = dls.one_batch()

### Create val, train and test data

In [ ]:
def acc_camvid(inp, targ):
  targ = targ.squeeze(1)
  mask = targ != void_code
  return (inp.argmax(dim=1)[mask]==targ[mask]).float().mean()

name2id = {'Background': 0, 'Point': 1, 'Void': 2}
void_code = name2id['Void']

### Create `UNet` Learner

In [ ]:
config = unet_config(self_attention=True, act_cls=Mish)
opt = ranger

# Initialiser un learner
learn = unet_learner(dls, resnet34, metrics=acc_camvid, config=config,
                     opt_func=opt, n_out=2)

### Training Model

In [ ]:
%%time
# learn.summary()
learn.lr_find()

In [ ]:
lr=1e-3
wd=1e-2
learn.fit_one_cycle(4, slice(lr), pct_start=0.9, wd=wd, cbs=WandbCallback())

In [ ]:
learn.fit_one_cycle(3, slice(lr), pct_start=0.9, wd=wd, cbs=WandbCallback())

### Save/Load Model

In [ ]:
learn.save('dartsify_ai')
!cp "models/dartsify_ai.pth" "drive/My Drive/Dartsify/models/"

### Make prediction or view results

In [ ]:
# Verify ground truth with predicted coordinates

learn.show_results(ds_idx=1, max_n=12, figsize=(20,14))

In [ ]:
%%time
preds, y, losses = learn.get_preds(with_loss=True)

# Top losses not implemented yet in fastai2

# interp = Interpretation.from_learner(learn)
# interp.plot_top_losses(5)

loss = pd.Series(losses)
loss.hist(bins=50)

In [ ]:
# Example Unet prediction mask

plt.figure(figsize = (10, 10))
plt.imshow(preds[0].numpy()[0])

### Prediction Overlay
* Using `Unet` Segmentation output, and `OpenCV` blob detector to find `(x, y)` coordinates.

In [ ]:
# Convert prediction to cv2 image
n = 0
img = preds[n].numpy()[0] * 256
orig_img = img.copy()

# Convert the grayscale image to binary image
retval, thresh = cv2.threshold(img, 180, 255, 0)
thresh = cv2.convertScaleAbs(thresh)

# Find contours in the binary image
contours = cv2.findContours(thresh, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
contours = imutils.grab_contours(contours)

for c in contours[1:]:
    # Calculate moments for each contour
    M = cv2.moments(c)
    cX = int(M["m10"] / M["m00"])
    cY = int(M["m01"] / M["m00"])
    print("Centroid located:", (cX, cY))

    # Transparency layer
    overlay = img.copy()
    output = img.copy()
    clr = 0
    cv2.circle(overlay, (cX, cY), 25, (clr, clr, clr), -1)
    alpha = 0.4
    cv2.addWeighted(overlay, alpha, output, 1 - alpha,0, output)

    # Dart point
    cv2.circle(output, (cX, cY), 25, (clr, clr, clr), 1)
    cv2.circle(output, (cX, cY), 4, (clr, clr, clr), 1)
    cv2.putText(output, f"{(cX, cY)}", (cX - 105, cY - 30), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, (clr, clr, clr), 1, cv2.LINE_AA)

val_img = dls.valid_ds[n][0]
val_img = cv2.cvtColor(np.array(val_img), cv2.COLOR_RGB2BGR)
val_img = val_img[:, :, ::-1].copy()

fig, ax = plt.subplots(1, 3)
ax[0].imshow(val_img), ax[1].imshow(orig_img), ax[2].imshow(output)
fig.set_size_inches(20, 5)
plt.tight_layout()

In [ ]:
pil_image = dls.valid_ds[n][0]
img = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)
img = img[:, :, ::-1].copy()

# Transparency layer
overlay = img.copy()
output = img.copy()
inner_colour = (255, 255, 255)
outer_colour = (255, 255, 255)
cv2.circle(overlay, (cX, cY), 25, outer_colour, -1)
alpha = 0.2
cv2.addWeighted(overlay, alpha, output, 1 - alpha,0, output)

# Dart point
cv2.circle(output, (cX, cY), 25, outer_colour, 1, cv2.LINE_AA)
cv2.circle(output, (cX, cY), 3, inner_colour, 1, cv2.LINE_AA)
cv2.putText(output, f"{(cX, cY)}", (cX - 105, cY - 30), cv2.FONT_HERSHEY_SIMPLEX,
            0.5, inner_colour, 1, cv2.LINE_AA)

plt.figure(figsize = (14, 14))
plt.imshow(output)
#plt.tight_layout()
plt.show()

In [ ]:
import random

In [ ]:
%%time
n = random.randint(0,90)
#n = 18
print('  Validation set:', n)

img = preds[n].numpy()[0] * 256
orig_img = img.copy()

# Convert the grayscale image to binary image
retval, thresh = cv2.threshold(img, 180, 255, 0)
thresh = cv2.convertScaleAbs(thresh)

# Find contours in the binary image
contours = cv2.findContours(thresh, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
contours = imutils.grab_contours(contours)

for c in contours[1:]:
    # Calculate moments for each contour
    M = cv2.moments(c)
    cX = int(M["m10"] / M["m00"])
    cY = int(M["m01"] / M["m00"])
    print("Centroid located:", (cX, cY))

    pil_image = dls.valid_ds[n][0]
    img = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)
    img = img[:, :, ::-1].copy()

    # Transparency layer
    overlay = img.copy()
    output = img.copy()
    inner_colour = (255, 255, 255)
    outer_colour = (255, 255, 255)
    cv2.circle(overlay, (cX, cY), 25, outer_colour, -1)
    alpha = 0.2
    cv2.addWeighted(overlay, alpha, output, 1 - alpha,0, output)

    # Dart point
    cv2.circle(output, (cX, cY), 25, outer_colour, 1, cv2.LINE_AA)
    cv2.circle(output, (cX, cY), 3, inner_colour, 1, cv2.LINE_AA)
    cv2.putText(output, f"{(cX, cY)}", (cX - 105, cY - 30), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, inner_colour, 1, cv2.LINE_AA)

plt.figure(figsize = (14, 14))
plt.imshow(output)
#plt.tight_layout()
plt.show()

### Profiling because it shouldn't take 500ms per frame

In [ ]:
!pip install line-profiler -q

In [ ]:
%load_ext line_profiler

In [ ]:
def predict_img(n):
  print('  Validation set:', n)
  img = preds[n].numpy()[0] * 256
  orig_img = img.copy()

  # Convert the grayscale image to binary image
  retval, thresh = cv2.threshold(img, 180, 255, 0)
  thresh = cv2.convertScaleAbs(thresh)

  # Find contours in the binary image
  contours = cv2.findContours(thresh, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
  contours = imutils.grab_contours(contours)

  for c in contours[1:]:
      # Calculate moments for each contour
      M = cv2.moments(c)
      cX = int(M["m10"] / M["m00"])
      cY = int(M["m01"] / M["m00"])
      print("Centroid located:", (cX, cY))

      pil_image = dls.valid_ds[n][0]
      img = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)
      img = img[:, :, ::-1].copy()

      # Transparency layer
      overlay = img.copy()
      output = img.copy()
      inner_colour = (255, 255, 255)
      outer_colour = (255, 255, 255)
      cv2.circle(overlay, (cX, cY), 25, outer_colour, -1)
      alpha = 0.2
      cv2.addWeighted(overlay, alpha, output, 1 - alpha,0, output)

      # Dart point
      cv2.circle(output, (cX, cY), 25, outer_colour, 1, cv2.LINE_AA)
      cv2.circle(output, (cX, cY), 3, inner_colour, 1, cv2.LINE_AA)
      cv2.putText(output, f"{(cX, cY)}", (cX - 105, cY - 30), cv2.FONT_HERSHEY_SIMPLEX,
                  0.5, inner_colour, 1, cv2.LINE_AA)

  plt.figure(figsize = (14, 14))
  plt.imshow(output)
  plt.show()


In [ ]:
%lprun -f predict_img predict_img(n=18)

95% of the 500ms taken to run is the three lines
```py
  plt.figure(figsize = (14, 14))
  plt.imshow(output)
  plt.show()
```
Running the function without `matplotlib` takes `0.023`s which is a frame rate of about `40` FPS ☑

### Export/Import the Model

`learn.export` saves both the model architecture, and the trained parameters.



In [ ]:
learn.export()

In [ ]:
%cp export.pkl "drive/My Drive/FlightVision/models/fv_nb016_export.pkl"

### Inference from Imported Model

In [ ]:
learn_inf = load_learner("drive/My Drive/FlightVision/models/fv_nb016_export.pkl")

In [ ]:
%%time
learn_inf.predict('dart2.jpeg')


This will create a file named 'export.pkl' in the directory where we were working that contains everything we need to deploy our model (the model, the weights but also some metadata like the classes or the transforms/normalization used).

You probably want to use CPU for inference, except at massive scale (and you almost certainly don't need to train in real-time). If you don't have a GPU that happens automatically. You can test your model on CPU like so:

In [ ]:
defaults.device = torch.device('cpu')

In [ ]:
learn = None

In [ ]:
# Clear cache and GPU memory
import gc
learn = None
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from fastai.utils.mem import gpu_mem_get_free_no_cache
free = gpu_mem_get_free_no_cache()
# the max size of bs depends on the available GPU RAM
if free > 8200: bs=3
else:           bs=1
print(f"using bs={bs}, have {free}MB of GPU RAM free")